In [1]:
# Install runtime dependencies
!pip install -q sewar scikit-image tifffile pillow

In [2]:
import os
import sys
import time
import glob
from pathlib import Path

import numpy as np
import scipy.io as sio
from PIL import Image

import torch
import torch.nn as nn

try:
    from skimage.metrics import structural_similarity
except Exception:
    from skimage.measure import compare_ssim as structural_similarity

print('✓ Imports ready')

✓ Imports ready


In [3]:
def get_lrhsi(img, degradation_mode):
    if degradation_mode == 0:
        scale_factor = 8
        dim = np.shape(img)
        img_down = np.zeros([dim[0], int(dim[1] / scale_factor), int(dim[2] / scale_factor)])
        img_rebuild = np.zeros(dim)
        kernel = np.array([[0.0067, 0.0094, 0.0118, 0.0131, 0.0131, 0.0118, 0.0094, 0.0067],
              [0.0094, 0.0131, 0.0164, 0.0183, 0.0183, 0.0164, 0.0131, 0.0094],
              [0.0118, 0.0164, 0.0205, 0.0229, 0.0229, 0.0205, 0.0164, 0.0118],
              [0.0131, 0.0183, 0.0229, 0.0256, 0.0256, 0.0229, 0.0183, 0.0131],
              [0.0131, 0.0183, 0.0229, 0.0256, 0.0256, 0.0229, 0.0183, 0.0131],
              [0.0118, 0.0164, 0.0205, 0.0229, 0.0229, 0.0205, 0.0164, 0.0118],
              [0.0094, 0.0131, 0.0164, 0.0183, 0.0183, 0.0164, 0.0131, 0.0094],
              [0.0067, 0.0094, 0.0118, 0.0131, 0.0131, 0.0118, 0.0094, 0.0067]])
        kernel = [kernel] * dim[0]
        for i in range(int(dim[1] / scale_factor)):
            for j in range(int(dim[2] / scale_factor)):
                img_down[:, i, j] = np.sum(np.sum(img[:, i * scale_factor:(i + 1) * scale_factor, j * scale_factor:(j + 1) * scale_factor] * kernel, axis=1), axis=1)
    elif degradation_mode == 1:
        scale_factor = 8
        dim = np.shape(img)
        img_down = np.zeros([dim[0], int(dim[1] / scale_factor), int(dim[2] / scale_factor)])
        img_rebuild = np.zeros(dim)
        kernel = np.ones([int(scale_factor), int(scale_factor)]) / (scale_factor ** 2)
        kernel = [kernel] * dim[0]
        for i in range(int(dim[1] / scale_factor)):
            for j in range(int(dim[2] / scale_factor)):
                img_down[:, i, j] = np.sum(np.sum(img[:, i * scale_factor:(i + 1) * scale_factor, j * scale_factor:(j + 1) * scale_factor] * kernel, axis=1), axis=1)
    elif degradation_mode == 2:
        scale_factor = 16
        dim = np.shape(img)
        img_down = np.zeros([dim[0], int(dim[1] / scale_factor), int(dim[2] / scale_factor)])
        img_rebuild = np.zeros(dim)
        kernel = np.ones([int(scale_factor), int(scale_factor)]) / (scale_factor ** 2)
        kernel = [kernel] * dim[0]
        for i in range(int(dim[1] / scale_factor)):
            for j in range(int(dim[2] / scale_factor)):
                img_down[:, i, j] = np.sum(np.sum(img[:, i * scale_factor:(i + 1) * scale_factor, j * scale_factor:(j + 1) * scale_factor] * kernel, axis=1), axis=1)
    else:
        scale_factor = 32
        dim = np.shape(img)
        img_down = np.zeros([dim[0], int(dim[1] / scale_factor), int(dim[2] / scale_factor)])
        img_rebuild = np.zeros(dim)
        kernel = np.ones([int(scale_factor), int(scale_factor)]) / (scale_factor ** 2)
        kernel = [kernel] * dim[0]
        for i in range(int(dim[1] / scale_factor)):
            for j in range(int(dim[2] / scale_factor)):
                img_down[:, i, j] = np.sum(np.sum(img[:, i * scale_factor:(i + 1) * scale_factor, j * scale_factor:(j + 1) * scale_factor] * kernel, axis=1), axis=1)

    for i in range(dim[0]):
        img_down_slice = Image.fromarray(img_down[i, :, :])
        img_rebuild[i, :, :] = img_down_slice.resize((dim[1], dim[2]), Image.BICUBIC)
    return img_rebuild.astype(np.float32)

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.prelu = nn.PReLU()
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = self.conv1(x)
        residual = self.bn1(residual)
        residual = self.prelu(residual)
        residual = self.conv2(residual)
        residual = self.bn2(residual)
        return x + residual

class Net(nn.Module):
    def __init__(self, HSI_num_residuals=12, RGB_num_residuals=12):
        super().__init__()
        self.input_1 = nn.Sequential(
            nn.Conv2d(31, 64, kernel_size=3, padding=1),
            nn.PReLU())
        self.residual_layers_1 = nn.Sequential(*[nn.Sequential(ResidualBlock(64)) for _ in range(HSI_num_residuals)])
        self.output_1 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64))

        self.input_2 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.PReLU())
        self.residual_layers_2 = nn.Sequential(*[nn.Sequential(ResidualBlock(64)) for _ in range(RGB_num_residuals)])
        self.output_2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64))

        self.layer_fusion = nn.Sequential(
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.PReLU(),
            nn.Conv2d(64, 31, kernel_size=3, padding=1))

    def forward(self, input_hsi, input_rgb):
        out1_1 = self.input_1(input_hsi)
        out2_1 = self.residual_layers_1(out1_1)
        out3_1 = self.output_1(out2_1)

        out1_2 = self.input_2(input_rgb)
        out2_2 = self.residual_layers_2(out1_2)
        out3_2 = self.output_2(out2_2)

        out4 = self.layer_fusion(torch.cat((out3_1, out3_2), 1))
        return torch.add(input_hsi, out4)

print('✓ TSFN model and degradation helper defined')

✓ TSFN model and degradation helper defined


In [4]:
def load_mat_kaggle(path):
    """Load Kaggle CAVE .mat or .tif and normalize to [0, 1]."""
    if path.endswith('.tif'):
        import tifffile as tiff
        img = tiff.imread(path).astype(np.float32)
        if img.ndim == 3 and img.shape[0] in [31, 34]:
            img = img.transpose(1, 2, 0)
        img_max = img.max()
        if img_max > 1.0:
            img = img / img_max
        return np.clip(img, 0.0, 1.0)

    mat_data = sio.loadmat(path)
    for key in ['hsi', 'gt', 'msi', 'X', 'ref']:
        if key in mat_data:
            arr = mat_data[key]
            break
    else:
        keys = [k for k in mat_data.keys() if not k.startswith('__')]
        arr = mat_data[keys[0]] if keys else None

    if arr is None:
        raise ValueError(f'No suitable key found in {path}')

    arr = np.asarray(arr, dtype=np.float32)
    if arr.ndim == 3 and arr.shape[0] < min(arr.shape[1], arr.shape[2]):
        arr = arr.transpose(1, 2, 0)

    arr_max = float(np.nanmax(arr)) if arr.size else 1.0
    if arr_max > 1.0:
        arr = arr / arr_max

    return np.clip(arr, 0.0, 1.0)

def compute_sam(image1, image2):
    image1 = np.asarray(image1)
    image2 = np.asarray(image2)
    if image1.ndim == 4:
        image1 = image1[0]
    if image2.ndim == 4:
        image2 = image2[0]
    if image1.ndim == 3 and image1.shape[0] < image1.shape[2]:
        image1 = image1.transpose(1, 2, 0)
    if image2.ndim == 3 and image2.shape[0] < image2.shape[2]:
        image2 = image2.transpose(1, 2, 0)
    h, w, c = image1.shape
    image1 = np.reshape(image1, (h * w, c))
    image2 = np.reshape(image2, (h * w, c))
    mole = np.sum(np.multiply(image1, image2), axis=1)
    image1_norm = np.sqrt(np.sum(np.square(image1), axis=1))
    image2_norm = np.sqrt(np.sum(np.square(image2), axis=1))
    deno = np.multiply(image1_norm, image2_norm)
    sam = np.rad2deg(np.arccos((mole + 1e-11) / (deno + 1e-11)))
    return np.mean(sam)

def compute_ergas(mse, out, sf=8):
    out = np.asarray(out)
    if out.ndim == 4:
        out = out[0]
    if out.ndim == 3 and out.shape[0] < out.shape[2]:
        out = out.transpose(1, 2, 0)
    h, w, c = out.shape
    out = np.reshape(out, (h * w, c))
    out_mean = np.mean(out, axis=0)
    mse = np.reshape(mse, (c, 1))
    out_mean = np.reshape(out_mean, (c, 1))
    ergas = 100.0 / float(sf) * np.sqrt(np.mean(mse / (out_mean ** 2 + 1e-12)))
    return ergas

def compute_psnr(image1, image2, data_range=1.0):
    image1 = np.asarray(image1, dtype=np.float32)
    image2 = np.asarray(image2, dtype=np.float32)
    mse = np.mean((image1 - image2) ** 2)
    if mse == 0:
        return float('inf')
    return 10.0 * np.log10((data_range ** 2) / mse)

def compute_ssim(image1, image2, data_range=1.0):
    image1 = np.asarray(image1)
    image2 = np.asarray(image2)
    if image1.ndim == 4:
        image1 = image1[0]
    if image2.ndim == 4:
        image2 = image2[0]
    if image1.ndim == 3 and image1.shape[0] < image1.shape[2]:
        image1 = image1.transpose(1, 2, 0)
    if image2.ndim == 3 and image2.shape[0] < image2.shape[2]:
        image2 = image2.transpose(1, 2, 0)
    h, w, c = image1.shape
    ssim_total = 0.0
    for i in range(c):
        ssim_total += structural_similarity(image1[:, :, i], image2[:, :, i], data_range=data_range)
    return ssim_total / c

print('✓ Helpers defined')

✓ Helpers defined


In [5]:
def run_tsfn_test(hsi_dir, rgb_dir, sf=8, weights=None, cuda=1):
    """Run TSFN directly on Kaggle CAVE .mat files."""
    degradation_mode = {8: 0, 16: 2, 32: 3}.get(sf, 0)
    if weights is None:
        weights = '/workspaces/hif-benchmarking/methods/_TSFN/models/ssfsr_9layers_epoch500.pkl'

    print('=' * 70)
    print(f'[TSFN] SF={sf}, degradation_mode={degradation_mode}')
    print(f'HSI dir: {hsi_dir}')
    print(f'RGB dir: {rgb_dir}')
    print(f'Weights: {weights}')

    device = torch.device('cuda' if cuda and torch.cuda.is_available() else 'cpu')
    checkpoint = torch.load(weights, map_location=device)
    model = Net(HSI_num_residuals=6, RGB_num_residuals=6)

    state_dict = checkpoint['state_dict']
    if any(k.startswith('module.') for k in state_dict.keys()):
        state_dict = {k.replace('module.', '', 1): v for k, v in state_dict.items()}

    model.load_state_dict(state_dict)
    model = model.to(device).eval()

    hsi_files = sorted(glob.glob(os.path.join(hsi_dir, '*.mat')))
    if not hsi_files:
        hsi_files = sorted(glob.glob(os.path.join(hsi_dir, '*/*.mat')))
    if not hsi_files:
        hsi_files = sorted(glob.glob(os.path.join(hsi_dir, '*.tif'))) + sorted(glob.glob(os.path.join(hsi_dir, '*/*.tif')))

    if not hsi_files:
        raise FileNotFoundError(f'No data files found in {hsi_dir}')

    results = {'psnr': [], 'ssim': [], 'sam': [], 'ergas': []}

    with torch.no_grad():
        for idx, hsi_path in enumerate(hsi_files):
            name = os.path.basename(hsi_path).replace('.mat', '').replace('.tif', '')
            hsi_full = load_mat_kaggle(hsi_path)

            if idx == 0:
                print(f'Validating first file: {name}, shape={hsi_full.shape}')

            if hsi_full.ndim == 4:
                hsi_full = hsi_full[0]
            if hsi_full.shape[2] > 31:
                hsi_full = hsi_full[:, :, :31]

            rgb_path = os.path.join(rgb_dir, os.path.basename(hsi_path))
            if os.path.exists(rgb_path):
                rgb_full = load_mat_kaggle(rgb_path)
                if rgb_full.ndim == 4:
                    rgb_full = rgb_full[0]
                if rgb_full.shape[2] > 3:
                    rgb_full = rgb_full[:, :, :3]
            else:
                idx_r, idx_g, idx_b = 23, 15, 7
                if hsi_full.shape[2] >= idx_r + 1:
                    rgb_full = np.stack([hsi_full[..., idx_r], hsi_full[..., idx_g], hsi_full[..., idx_b]], axis=-1)
                else:
                    rgb_full = np.tile(hsi_full[..., :1], (1, 1, 3))

            hsi_chw = hsi_full.transpose(2, 0, 1).astype(np.float32)
            rgb_chw = rgb_full.transpose(2, 0, 1).astype(np.float32)
            lr_hsi_chw = get_lrhsi(hsi_chw, degradation_mode).astype(np.float32)
            lr_hsi_chw = np.clip(lr_hsi_chw / max(lr_hsi_chw.max(), 1.0), 0.0, 1.0)

            lr_hsi_tensor = torch.from_numpy(lr_hsi_chw).unsqueeze(0).to(device)
            rgb_tensor = torch.from_numpy(rgb_chw).unsqueeze(0).to(device)

            start = time.time()
            pred = model(lr_hsi_tensor, rgb_tensor)
            elapsed = time.time() - start

            pred_np = pred[0].permute(1, 2, 0).cpu().numpy()
            gt_np = hsi_chw.transpose(1, 2, 0)
            pred_np = np.clip(pred_np, 0.0, 1.0)
            gt_np = np.clip(gt_np, 0.0, 1.0)

            psnr = compute_psnr(gt_np, pred_np, data_range=1.0)
            sam = compute_sam(gt_np, pred_np)
            ssim = compute_ssim(gt_np, pred_np, data_range=1.0)
            mse_per_channel = np.mean((gt_np - pred_np) ** 2, axis=(0, 1))
            ergas = compute_ergas(mse_per_channel, pred_np, sf=sf)

            results['psnr'].append(psnr)
            results['ssim'].append(ssim)
            results['sam'].append(sam)
            results['ergas'].append(ergas)

            print(f'  {idx + 1}/{len(hsi_files)}: {name}: PSNR={psnr:.2f} SAM={sam:.2f}° ERGAS={ergas:.3f} SSIM={ssim:.4f} ({elapsed:.2f}s)')

    avg = {k: float(np.mean(v)) for k, v in results.items()}
    print('\n' + '=' * 70)
    print(f'AVERAGE (SF={sf}, n={len(hsi_files)}):')
    print(f"  PSNR: {avg['psnr']:.2f}")
    print(f"  SAM:  {avg['sam']:.2f}°")
    print(f"  ERGAS: {avg['ergas']:.3f}")
    print(f"  SSIM: {avg['ssim']:.4f}")
    print('=' * 70)
    return avg

print('✓ TSFN runner defined')

✓ TSFN runner defined


In [6]:
TEST_CONFIGS = {
    'SF8': {
        'hsi_dir': '/kaggle/input/cave-dataset-2/Data/Test/HSI',
        'rgb_dir': '/kaggle/input/cave-dataset-2/Data/Test/RGB',
        'sf': 8
    }
}

WEIGHTS = '/kaggle/input/model-tsfn-epoch500/tensorflow2/default/1/ssfsr_9layers_epoch500.pkl'
print('✓ Test config loaded')
for name, cfg in TEST_CONFIGS.items():
    print(f"{name}: sf={cfg['sf']}, hsi_dir={cfg['hsi_dir']}")

✓ Test config loaded
SF8: sf=8, hsi_dir=/kaggle/input/cave-dataset-2/Data/Test/HSI


In [7]:
result_sf8 = run_tsfn_test(
    hsi_dir=TEST_CONFIGS['SF8']['hsi_dir'],
    rgb_dir=TEST_CONFIGS['SF8']['rgb_dir'],
    sf=TEST_CONFIGS['SF8']['sf'],
    weights=WEIGHTS,
    cuda=1
)

[TSFN] SF=8, degradation_mode=0
HSI dir: /kaggle/input/cave-dataset-2/Data/Test/HSI
RGB dir: /kaggle/input/cave-dataset-2/Data/Test/RGB
Weights: /kaggle/input/model-tsfn-epoch500/tensorflow2/default/1/ssfsr_9layers_epoch500.pkl
Validating first file: jelly_beans, shape=(512, 512, 31)
  1/12: jelly_beans: PSNR=42.42 SAM=3.39° ERGAS=0.704 SSIM=0.9928 (0.76s)
  2/12: oil_painting: PSNR=43.84 SAM=3.02° ERGAS=0.794 SSIM=0.9918 (0.00s)
  3/12: paints: PSNR=43.02 SAM=2.39° ERGAS=0.475 SSIM=0.9942 (0.00s)
  4/12: photo_and_face: PSNR=43.44 SAM=4.68° ERGAS=1.551 SSIM=0.9906 (0.00s)
  5/12: pompoms: PSNR=45.65 SAM=2.07° ERGAS=0.422 SSIM=0.9935 (0.00s)
  6/12: real_and_fake_apples: PSNR=55.30 SAM=2.26° ERGAS=0.479 SSIM=0.9981 (0.00s)
  7/12: real_and_fake_peppers: PSNR=53.21 SAM=1.79° ERGAS=0.334 SSIM=0.9978 (0.00s)
  8/12: sponges: PSNR=45.74 SAM=1.64° ERGAS=0.330 SSIM=0.9942 (0.00s)
  9/12: stuffed_toys: PSNR=44.37 SAM=3.26° ERGAS=0.650 SSIM=0.9944 (0.00s)
  10/12: superballs: PSNR=48.90 SAM=3.